### Day 10 · OOP 多态+契约 (L3)

今天进入 OOP 最强大的武器:**多态**与**契约**。
你会学会:用同一接口处理不同对象、用 abc 强制子类履约。

> 叙事锚点:电商 v2 —— Payment(abc) 支付系统
> 教学法:本节是 **NCDL**(负案例驱动)的完整落地

> Step 0:痛点 -- 120 行 if-elif 的灾难

    def pay(payment_type, amount):
        if payment_type == "alipay":
            print(f"支付宝支付 {amount}")
        elif payment_type == "wechat":
            print(f"微信支付 {amount}")
        # 每新增一种都要改这里!

3 个致命问题:新增=改核心/函数越来越长/测试越来越难
**今天的目标**: pay 函数缩小到 4 行,新增支付方式只需加文件。

#### 鸭子类型:Duck Typing 多态

> **类比**  
'像鸭子就是鸭子':走路像、叫声像,它就是鸭子,不管它继承谁。

> **解释**  
Python 不关注对象是哪个类,只关心 '有没有这个方法'。

In [ ]:
class Alipay:
    def execute(self, amount):
        return f"支付宝支付 {amount} 元"

class WeChatPay:
    def execute(self, amount):
        return f"微信支付 {amount} 元"

def checkout(cart_total, payment):
    return payment.execute(cart_total)

for p in [Alipay(), WeChatPay()]:
    print(checkout(99.0, p))

> **逐行解剖**  
① 三个类**没有继承同一个父类**  
② 但它们都有 `execute` 方法 → 鸭子类型  
③ `checkout` **不关心类型**,只关心有没有 `execute`  
④ 完全去掉 if-elif!

**练习** — 不继承也能多态

定义三个媒体类 `Audio`、`Video`、`Image`,
各自实现 `play()` 方法。
再写 `play_media(media)` 调用 `media.play()`。

In [ ]:
# ============ 学员代码区 ============
class Audio: pass

# class Video: pass
# class Image: pass
# def play_media(media): ...
pass

# ============ 参考答案 ============
class Audio:
    def play(self): return "播放音频"
class Video:
    def play(self): return "播放视频"
class Image:
    def play(self): return "展示图片"

def play_media(media):
    print(media.play())

for m in [Audio(), Video(), Image()]:
    play_media(m)

> NCDL Break It #1:鸭子类型的代价

鸭子类型的致命弱点:**漏写方法不会提前报错**。

In [ ]:
# ============ BREAK IT 演示 ============
class BrokenAlipay:
    # 注意:execute 方法被漏写了!
    pass

def checkout(total, payment):
    return payment.execute(total)

alipay = BrokenAlipay()
print("创建成功,一切看似正常...")
try:
    checkout(99.0, alipay)
except AttributeError as e:
    print(f"报错: {e}")
    print("错误在运行中才暴露,难以追溯!")
# ============ END BREAK IT ============

**诊断**:鸭子类型的弱点 —— bug 难追溯。
**结论**:个人脚本用鸭子类型 OK,但团队项目需要**更强的保护**。

#### abc.ABC 抽象基类 + @abstractmethod

> **类比**  
国家发布 USB-C 接口规范:每个厂商的设备必须有 type-c 口。

> **解释**  
- `abc.ABC` 作为父类 → 类成为抽象基类
- `@abstractmethod` 装饰的方法 → 子类**必须实现**
- 漏写 → **实例化时报 TypeError**

In [ ]:
import abc

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount): ...

try:
    p = Payment()
except TypeError as e:
    print(f"报错: {e}")

class Alipay(Payment):
    def execute(self, amount):
        return f"支付宝支付 {amount} 元"

alipay = Alipay()
print(alipay.execute(99.0))

> **逐行解剖**  
① `class Payment(abc.ABC)` 成为抽象基类  
② `@abc.abstractmethod` 标记契约  
③ `Payment()` → `TypeError`  
④ `Alipay(Payment)` 实现 execute → 可实例化

**练习** — Plugin 抽象类

定义抽象类 `Plugin`,包含抽象方法 `name()` 和 `execute(data)`。
尝试直接实例化 `Plugin` 感受报错。
再创建空子类 `MyPlugin` 不实现方法,同样尝试实例化。

In [ ]:
# ============ 学员代码区 ============
import abc

class Plugin(abc.ABC):
    # 请定义 name() 和 execute(data) 抽象方法
    pass

# try:
#     p = Plugin()
# except TypeError as e:
#     print(f"报错: {e}")
pass

# ============ 参考答案 ============
import abc

class Plugin(abc.ABC):
    @abc.abstractmethod
    def name(self): ...
    @abc.abstractmethod
    def execute(self, data): ...

try:
    p = Plugin()
except TypeError as e:
    print(f"报错: {e}")

class MyPlugin(Plugin):
    pass

try:
    mp = MyPlugin()
except TypeError as e:
    print(f"子类报错: {e}")

> NCDL Break It #2:忘记继承 abc.ABC

场景:你用了 @abstractmethod,但忘了继承 abc.ABC。
'能跑',但漏写 execute 不报错 → 鸭子类型的弱点又回来了!

In [ ]:
# ============ BREAK IT 演示 ============
import abc

class BrokenPayment:  # 没有 abc.ABC! BREAK IT!
    @abc.abstractmethod
    def execute(self, amount): ...

class Alipay(BrokenPayment):
    pass  # 漏写 execute!

alipay = Alipay()  # 竟然不报错!
print("没报错!但 execute 不存在,运行时才爆炸")
try:
    alipay.execute(99)
except AttributeError as e:
    print(f"运行时报错: {e}")
# ============ END BREAK IT ============

**结论**:`@abstractmethod` + `abc.ABC` 必须一起用。

#### 接口概念(团队协作契约)

> **解释**  
- 接口 = 只包含抽象方法的抽象类
- Python 用'全抽象方法的 ABC'模拟接口

In [ ]:
import abc

class Notifier(abc.ABC):
    @abc.abstractmethod
    def send(self, msg): ...
    @abc.abstractmethod
    def channel(self): ...

class EmailNotifier(Notifier):
    def send(self, msg): return f"邮件: {msg}"
    def channel(self): return "email"

class SMSNotifier(Notifier):
    def send(self, msg): return f"短信: {msg}"
    def channel(self): return "sms"

for n in [EmailNotifier(), SMSNotifier()]:
    print(f"[{n.channel()}] {n.send('订单已发货')}")

**练习** — Serializer 接口

定义接口 `Serializer`,抽象方法 `serialize(obj)` 和 `deserialize(text)`。
子类 `JsonSerializer` 简单模拟格式。

In [ ]:
# ============ 学员代码区 ============
import abc

class Serializer(abc.ABC):
    # 请定义 serialize 抽象方法
    pass

# class JsonSerializer(Serializer): ...
pass

# ============ 参考答案 ============
import abc

class Serializer(abc.ABC):
    @abc.abstractmethod
    def serialize(self, obj): ...
    @abc.abstractmethod
    def deserialize(self, text): ...

class JsonSerializer(Serializer):
    def serialize(self, obj):
        return "{data: " + str(obj) + "}"
    def deserialize(self, text):
        return text

s = JsonSerializer()
print(s.serialize("hello"))

#### 综合练习:Payment 支付系统(L3 完整落地)

目标:把 Day09 末尾那个有 if-elif 味的系统,重构为多态版本。

> - `Payment(abc.ABC)` 定义契约
> - `checkout(cart_total, payment)` **不使用 if/elif/isinstance/type**
> - 漏写 `execute` 时在实例化阶段就报 TypeError
> - 新增支付方式只需**添加一个文件**

In [ ]:
# ============ 学员代码区 ============
import abc

class Payment(abc.ABC):
    # 定义 execute 抽象方法
    pass

class Alipay(Payment): pass
class WeChatPay(Payment): pass
class ApplePay(Payment): pass

def checkout(cart_total, payment):
    # 不超过 4 行,不使用 if/elif/isinstance/type
    pass

payments = [Alipay(), WeChatPay(), ApplePay()]
for p in payments:
    print(checkout(99.0, p))
pass

# ============ 参考答案 ============
import abc

class Payment(abc.ABC):
    @abc.abstractmethod
    def execute(self, amount: float) -> bool: ...

class Alipay(Payment):
    def execute(self, amount):
        print(f"支付宝支付 {amount} 元")
        return True

class WeChatPay(Payment):
    def execute(self, amount):
        print(f"微信支付 {amount} 元")
        return True

class ApplePay(Payment):
    def execute(self, amount):
        print(f"Apple Pay {amount} 元")
        return True

def checkout(cart_total, payment):
    if cart_total <= 0:
        print("购物车为空")
        return False
    return payment.execute(cart_total)

for p in [Alipay(), WeChatPay(), ApplePay()]:
    checkout(99.0, p)

**今日小结**

| 概念 | 解决的痛点 |
| --- | --- |
| 鸭子类型 | 不写继承也能多态 |
| abc.ABC 抽象基类 | 漏实现立刻报错 |
| @abstractmethod | 把规范写进代码 |
| 接口 | 多人协作统一方法签名 |

**更多练习**
- 当堂练:course/lesson10/in_class/practice01-06.py
- 课后作业:course/lesson10/homework/task01-03.py